# KV Cache Compression — Evaluation & Analysis
### Mistral-7B-Instruct-v0.3 · H2O · StreamingLLM · LongBench

---

Run top-to-bottom. Each section builds on the previous one.

```
Step 1  Install / verify dependencies
Step 2  Background: how each method works
Step 3  Load model
Step 4  Visual demo — see the KV cache in action on a short prompt
Step 5  LongBench smoke-test (5 samples × 3 tasks, ~10 min on an A100)
Step 6  Full LongBench evaluation (uncomment when ready, ~1-2 h on A100)
Step 7  Results — tables, bar chart, efficiency–accuracy curve, latency
Step 8  Discussion template
```

**Hardware guide for Mistral-7B-Instruct-v0.3 bfloat16 (~14 GB):**

| Hardware | Notes |
|---|---|
| NVIDIA A100 / H100 | Run as-is |
| RTX 3090 / 4090 (24 GB) | Run as-is |
| T4 / RTX 3080 (16 GB) or < 16 GB VRAM | Set `LOAD_IN_4BIT = True` in Step 3 |
| Apple M-series (≥ 16 GB unified RAM) | Run as-is (MPS) |
| CPU only | Smoke-test only; set `DEVICE_MAP = "cpu"` |

---
## Step 1 — Install dependencies

Run once, then **restart the kernel** before proceeding.

In [14]:
import subprocess, sys

# transformers 4.44.2 is pinned because:
#   - SinkCache (used by StreamingLLM) was removed in transformers 5.0
#   - DynamicCache exposed key_cache / value_cache lists in 4.x (H2O depends on this)
#   - Mistral-7B-Instruct-v0.3 is fully supported in 4.44.2
packages = [
    "transformers==4.44.2",
    "datasets>=2.18.0",
    "accelerate>=0.27.0",
    "sentencepiece",
    "tqdm",
    "rouge-score",
    "matplotlib",
    "pandas",
    "seaborn",
    "ipywidgets",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print("Done. Restart the kernel now if this was the first install.")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Done. Restart the kernel now if this was the first install.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
# Verify pinned version
import transformers
assert transformers.__version__.startswith("4.44"), (
    f"Expected transformers 4.44.x, got {transformers.__version__}.\n"
    "Run the install cell above and restart the kernel."
)
print(f"transformers {transformers.__version__}  OK")

transformers 4.44.2  OK


---
## Step 2 — Background

### The problem: KV cache memory

During autoregressive generation, every token's key and value projections must be cached so subsequent tokens can attend to them without recomputing. The cache grows as:

```
memory = 2 × layers × heads × head_dim × seq_len × bytes_per_element
```

For Mistral-7B-Instruct-v0.3 at bfloat16: **~0.13 MB per token**. A 4096-token context = ~512 MB — just for the cache.

---

### H2O — Heavy-Hitter Oracle (NeurIPS 2023)
Paper: https://arxiv.org/abs/2306.14048 | Reference impl: [KVCache-Factory](https://github.com/Zefan-Cai/KVCache-Factory)

**Observation:** Attention score distributions are highly skewed. A small subset of tokens — "heavy hitters" — consistently absorbs most of the attention mass across all layers and heads, regardless of position.

**Policy (per layer):**
```
budget B = HH + Recent
           |       └── last R tokens, always kept (recency window)
           └────────── top-K tokens by cumulative attention score

If cache_len > B:  drop all tokens NOT in {top-K} ∪ {last-R}
```

**Our implementation** (adapted from KVCache-Factory's `H2OKVCluster`):
- `patch_model_for_h2o()` monkey-patches each attention layer's `forward` to intercept softmax weights.
- `H2OCache` (subclasses `DynamicCache`) accumulates per-token importance scores and calls `_evict_layer()` whenever the cache exceeds budget.
- Requires `attn_implementation="eager"` — SDPA and Flash-Attention do not return softmax weights even with `output_attentions=True`.

---

### StreamingLLM — Attention Sinks (ICLR 2024)
Paper: https://arxiv.org/abs/2309.17453

**Observation:** The very first few tokens in any sequence act as "attention sinks" — they receive large attention weights regardless of semantic content. This happens because softmax must sum to 1; when no token is relevant, the model dumps attention mass onto position 0.

**Policy:**
```
Cache layout:  [ sink_0 … sink_{S-1} | oldest_recent … newest_recent ]
                       S tokens                    W tokens
Budget B = S + W.  Evict oldest non-sink when len > B.
```

**Our implementation:** thin wrapper around HuggingFace's built-in `SinkCache`. No model patching needed.

---

| | H2O | StreamingLLM |
|---|---|---|
| Eviction criterion | Cumulative attention score | FIFO (oldest non-sink) |
| Attention backend | eager only | any |
| Tracks important tokens globally | ✅ | ❌ |
| Overhead | score accumulation | none |
| Good for retrieval / multi-hop QA | ✅ | ❌ |
| Good for streaming / summarisation | ✅ | ✅ |

---
## Step 3 — Load Mistral-7B-Instruct-v0.3

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # make src/ importable from notebooks/

import torch

# ── Change these if needed ────────────────────────────────────────────────────
MODEL_NAME   = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_SLUG   = "qwen2.5-1.5b"   # used as results subdirectory — change per model
LOAD_IN_4BIT = False    # True if VRAM < 8 GB (requires: pip install bitsandbytes)

if torch.cuda.is_available():
    DEVICE     = "cuda"
    DEVICE_MAP = "auto"
elif torch.backends.mps.is_available():
    DEVICE     = "mps"
    DEVICE_MAP = "mps"
else:
    DEVICE     = "cpu"
    DEVICE_MAP = "cpu"

print(f"Device     : {DEVICE}")
print(f"Model      : {MODEL_NAME}")
print(f"Model slug : {MODEL_SLUG}")
if DEVICE == "cuda":
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  ({p.total_memory/1e9:.1f} GB)")

Device     : mps
Model      : Qwen/Qwen2.5-1.5B-Instruct
Model slug : qwen2.5-1.5b


In [3]:
from src.models.patch import load_model_and_tokenizer

# Always load with eager attention — required for H2O to capture softmax weights.
# Full and StreamingLLM work fine with eager too, so one load covers all methods.
model, tokenizer = load_model_and_tokenizer(
    model_name=MODEL_NAME,
    attn_implementation="eager",
    device_map=DEVICE_MAP,
    torch_dtype=torch.bfloat16,
    load_in_4bit=LOAD_IN_4BIT,
)

n_layers = model.config.num_hidden_layers
n_params  = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model  : {MODEL_NAME}")
print(f"Params : {n_params:.2f} B")
print(f"Layers : {n_layers}")

Model  : Qwen/Qwen2.5-1.5B-Instruct
Params : 1.54 B
Layers : 28


---
## Step 4 — Visual demo: what each method does to the KV cache

We run the same short prompt through all three methods and inspect:
1. The final KV-cache length at layer 0 (is the budget respected?)
2. The generated text (does quality hold up?)

In [4]:
from src.kv_cache.h2o import H2OCache, patch_model_for_h2o
from src.kv_cache.streaming_llm import StreamingLLMCache
from src.models.patch import apply_kv_method

DEMO_PROMPT = (
    "[INST] Context: Transformer models use self-attention, which has quadratic "
    "O(N²) complexity in sequence length N. During autoregressive decoding the KV "
    "cache grows by one row per generated token, consuming GPU memory linearly with "
    "sequence length. This becomes the dominant bottleneck for long-context inference.\n\n"
    "Question: What is the primary memory bottleneck during long-context LLM inference, "
    "and why does it scale linearly? [/INST]"
)

inputs   = tokenizer(DEMO_PROMPT, return_tensors="pt").to(DEVICE)
n_prompt = inputs["input_ids"].shape[-1]
print(f"Prompt length : {n_prompt} tokens")

# 50% budget for demo so the difference is visible on a short prompt
BUDGET = max(8, n_prompt // 2)
HH     = BUDGET // 2
REC    = BUDGET - HH
print(f"Demo budget   : {BUDGET} tokens ({HH} HH + {REC} recent)")

Prompt length : 86 tokens
Demo budget   : 43 tokens (21 HH + 22 recent)


In [5]:
# Patch model for H2O (safe to call multiple times — patches are idempotent once applied)
apply_kv_method(model, "h2o", hh_ratio=0.25, recent_ratio=0.25, max_seq_len=n_prompt)

@torch.inference_mode()
def demo_run(label, cache_obj, gen_tokens=50):
    gen_kwargs = dict(max_new_tokens=gen_tokens, do_sample=False, use_cache=True)
    if cache_obj is not None:
        gen_kwargs["past_key_values"] = cache_obj

    out  = model.generate(**inputs, **gen_kwargs)
    text = tokenizer.decode(out[0, n_prompt:], skip_special_tokens=True).strip()

    # Read final cache size from layer 0
    if cache_obj is not None and getattr(cache_obj, "key_cache", None):
        kv_len  = cache_obj.key_cache[0].shape[-2]
        budget  = getattr(cache_obj, "budget", "?")
        kv_desc = f"{kv_len} / {budget} tokens used"
    else:
        kv_desc = f"{out.shape[-1]} tokens (full, no eviction)"


    print(f"\n{'─'*62}")
    print(f"  Method : {label}")
    print(f"  KV len : {kv_desc}")
    print(f"  Output : {text!r}")

demo_run("Full attention",                              None)
demo_run(f"H2O  ({HH} HH + {REC} recent)",            H2OCache(hh_size=HH, recent_size=REC))
demo_run(f"StreamingLLM  (4 sinks + {BUDGET-4} win)", StreamingLLMCache(sink_size=4, window_size=BUDGET-4))

/Users/jaisu/Projects/Kv_caching/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/Users/jaisu/Projects/Kv_caching/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/Users/jaisu/Projects/Kv_caching/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(



──────────────────────────────────────────────────────────────
  Method : Full attention
  KV len : 136 tokens (full, no eviction)
  Output : 'The primary memory bottleneck during long-context language model (LLM) inference is the growing KV (Key-Value) cache used for attention mechanisms. This scaling occurs because each generated token requires a new row of the KV cache to be updated, leading to'

──────────────────────────────────────────────────────────────
  Method : H2O  (21 HH + 22 recent)
  KV len : 43 / 43 tokens used
  Output : 'The primary memory bottleneck during long-context language model (LLM) inference is the growing size of the key-value pairs (KV pairs) in the Key Value Memory (KVM). This KVM stores information about the context used to generate a particular output'

──────────────────────────────────────────────────────────────
  Method : StreamingLLM  (4 sinks + 39 win)
  KV len : 39 / 43 tokens used
  Output : 'The primary memory bottleneck during long-context lan

**What to observe:**
- **Full**: KV len grows to prompt + generated tokens — no eviction.
- **H2O**: Stays at or below `budget`. Semantically important tokens survive; output should still be coherent.
- **StreamingLLM**: Stays at or below `budget`. First 4 sink tokens + sliding window. May miss mid-prompt context on long inputs.

The gap between methods widens dramatically on real LongBench inputs (2000–4000 tokens).

---
## Step 5 — LongBench smoke-test  (5 samples × 3 tasks)

Runs 3 methods at a **20 % cache budget** on a small slice of LongBench.  
Expected runtime: **~10 min on A100**, ~25 min on RTX 4090.

In [ ]:
from src.eval.longbench import LongBenchEvaluator

# ── Config ────────────────────────────────────────────────────────────────────
SMOKE_TASKS = [
    "narrativeqa",
    "qasper",
    "multifieldqa_en",
    "gov_report",
    "hotpotqa",
    "2wikimqa",
]
NUM_SAMPLES = 50      # examples per task
MAX_SEQ_LEN = 8192    # max input tokens
CACHE_RATIO = 0.20    # 20% KV budget
OUTPUT_DIR  = f"../results/{MODEL_SLUG}/smoke"

METHOD_CFGS = [
    ("full",          {}),
    ("h2o",           {"hh_ratio": CACHE_RATIO/2, "recent_ratio": CACHE_RATIO/2}),
    ("streaming_llm", {"cache_ratio": CACHE_RATIO, "sink_size": 4}),
]

In [ ]:
import time

smoke_results = {}

for method, kw in METHOD_CFGS:
    print(f"\n{'═'*55}\n  ▶  {method.upper()}\n{'═'*55}")
    apply_kv_method(model, method=method, max_seq_len=MAX_SEQ_LEN, **kw)

    ev = LongBenchEvaluator(
        model=model, tokenizer=tokenizer, method=method,
        tasks=SMOKE_TASKS, max_length=MAX_SEQ_LEN,
        num_samples=NUM_SAMPLES,
        output_dir=OUTPUT_DIR, device=DEVICE,
    )
    t0 = time.perf_counter()
    smoke_results[method] = ev.run()
    print(f"  ✓  {time.perf_counter()-t0:.1f}s")

print("\nEvaluation complete.")

In [11]:
import pandas as pd

rows = [
    {"method": m, "task": t,
     "score (%)": r["score"], "latency (s)": round(r["avg_latency_sec"], 3)}
    for m, td in smoke_results.items()
    for t, r in td.items()
]
df_smoke = pd.DataFrame(rows)

print("\n── Score (%) ──")
display(df_smoke.pivot(index="task", columns="method", values="score (%)").round(2))

print("\n── Avg latency per sample (s) ──")
display(df_smoke.pivot(index="task", columns="method", values="latency (s)").round(3))


── Score (%) ──


method,full,h2o,streaming_llm
task,,,
hotpotqa,14.67,14.67,14.67
multifieldqa_en,29.99,27.74,21.09
qasper,14.09,11.05,13.30



── Avg latency per sample (s) ──


method,full,h2o,streaming_llm
task,,,
hotpotqa,2.275,3.470,1.309
multifieldqa_en,13.187,5.352,4.302
qasper,19.516,9.010,2.924


---
## Step 6 — Full LongBench evaluation

Evaluates **3 methods × 3 budgets × 7 tasks** on the full test sets.  
~30–45 min per (method, budget) pair on an A100.

**Uncomment the cell below when ready.**

In [ ]:
from src.eval.longbench import LongBenchEvaluator
from src.models.patch import apply_kv_method

FULL_TASKS = [
    "narrativeqa",
    "qasper",
    "multifieldqa_en",
    "gov_report",
    "hotpotqa",
    "2wikimqa",
]
FULL_DIR       = f"../results/{MODEL_SLUG}"
MAX_SEQ        = 8192
MAX_NEW        = 64
N_SAMPLES      = 50

# Budget grid
BUDGETS = [
    ("10pct", 0.05, 0.05, 0.10),
    ("20pct", 0.10, 0.10, 0.20),
    ("50pct", 0.25, 0.25, 0.50),
]

all_results = {}

# Baseline — run once
apply_kv_method(model, "full")
ev = LongBenchEvaluator(model=model, tokenizer=tokenizer, method="full",
                        tasks=FULL_TASKS, max_length=MAX_SEQ, max_new_tokens=MAX_NEW,
                        num_samples=N_SAMPLES, output_dir=FULL_DIR, device=DEVICE)
all_results["full"] = ev.run()

# Compression methods at each budget
for tag, hh, rec, ratio in BUDGETS:
    for method, kw in [
        ("h2o",           {"hh_ratio": hh,    "recent_ratio": rec}),
        ("streaming_llm", {"cache_ratio": ratio, "sink_size": 4}),
    ]:
        run_id = f"{method}_{tag}"
        apply_kv_method(model, method, max_seq_len=MAX_SEQ, **kw)
        ev = LongBenchEvaluator(model=model, tokenizer=tokenizer, method=run_id,
                                tasks=FULL_TASKS, max_length=MAX_SEQ, max_new_tokens=MAX_NEW,
                                num_samples=N_SAMPLES, output_dir=FULL_DIR, device=DEVICE)
        all_results[run_id] = ev.run()

---
## Step 7 — Results analysis

Loads every `*.json` file from `../results/` and generates four views:
1. Score table with Δ vs full baseline
2. Per-task grouped bar chart
3. Efficiency–accuracy trade-off curve (score vs budget %)
4. Latency comparison

In [ ]:
import json, pathlib, re
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.15)

# ── Choose what to load ───────────────────────────────────────────────────────
# Single model:   RESULTS_DIR = pathlib.Path(f"../results/{MODEL_SLUG}")
# Compare models: RESULTS_DIR = pathlib.Path("../results")   (loads all subdirs)
RESULTS_DIR = pathlib.Path(f"../results/{MODEL_SLUG}")

records = []
for f in sorted(RESULTS_DIR.rglob("*.json")):
    if f.name.startswith(("summary", "benchmark")):
        continue
    try:
        d = json.loads(f.read_text())
        # Infer model from parent directory name if comparing across models
        model_tag = f.parts[-3] if RESULTS_DIR == pathlib.Path("../results") else MODEL_SLUG
        records.append({
            "model":   model_tag,
            "method":  d["method"],
            "task":    d["dataset"],
            "score":   d["score"],
            "latency": d["avg_latency_sec"],
            "metric":  d["metric"],
            "n":       d["num_samples"],
        })
    except Exception:
        pass

df = pd.DataFrame(records)
if df.empty:
    print("No results yet. Run §5 (smoke-test) or §6 (full eval) first.")
else:
    print(f"Loaded {len(df)} rows")
    print(f"Models  : {sorted(df['model'].unique())}")
    print(f"Methods : {sorted(df['method'].unique())}")

In [ ]:
# ── 1. Score table with Δ vs full ─────────────────────────────────────────────
if not df.empty:
    pivot = df.pivot_table(index="task", columns="method", values="score", aggfunc="mean")
    if "full" in pivot.columns:
        for col in [c for c in pivot.columns if c != "full"]:
            pivot[f"Δ {col}"] = (pivot[col] - pivot["full"]).round(2)
    display(pivot.round(2))

In [ ]:
# ── 2. Per-task grouped bar chart ─────────────────────────────────────────────
if not df.empty:
    methods = sorted(df["method"].unique())
    tasks   = sorted(df["task"].unique())
    x, w    = np.arange(len(tasks)), 0.8 / max(len(methods), 1)
    colors  = sns.color_palette("muted", n_colors=len(methods))

    fig, ax = plt.subplots(figsize=(14, 5))
    for i, m in enumerate(methods):
        sub    = df[df["method"] == m].set_index("task")
        vals   = [sub.loc[t, "score"] if t in sub.index else 0 for t in tasks]
        offset = x + i*w - (len(methods)-1)*w/2
        bars   = ax.bar(offset, vals, w*0.9, label=m, color=colors[i])
        ax.bar_label(bars, fmt="%.1f", fontsize=7, padding=2)

    ax.set_xticks(x); ax.set_xticklabels(tasks, rotation=25, ha="right")
    ax.set_ylabel("Score (%)"); ax.set_ylim(0, 110)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100))
    ax.set_title("LongBench: Per-task performance by KV compression method")
    ax.legend(title="Method", bbox_to_anchor=(1.01, 1), loc="upper left")
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "bar_scores.pdf", bbox_inches="tight")
    plt.show(); print("Saved → results/bar_scores.pdf")

In [ ]:
# ── 3. Efficiency–accuracy trade-off curve ────────────────────────────────────
# Expects method names like 'h2o_10pct', 'h2o_20pct', 'streaming_llm_50pct', 'full'
if not df.empty:
    trade = []
    for m in df["method"].unique():
        macro = df[df["method"] == m]["score"].mean()
        pct   = re.search(r"(\d+)pct", m)
        trade.append({
            "base":   re.sub(r"_\d+pct", "", m),
            "budget": int(pct.group(1)) if pct else (100 if m == "full" else None),
            "macro":  macro,
        })
    df_trade = pd.DataFrame(trade).dropna(subset=["budget"])

    if not df_trade.empty:
        fig, ax = plt.subplots(figsize=(7, 4))
        for base, grp in df_trade.groupby("base"):
            g = grp.sort_values("budget")
            ax.plot(g["budget"], g["macro"], marker="o", label=base)
        ax.set_xlabel("Cache budget (% of sequence length)")
        ax.set_ylabel("Macro-avg score (%)")
        ax.set_title("Efficiency–Accuracy trade-off")
        ax.legend(); fig.tight_layout()
        fig.savefig(RESULTS_DIR / "tradeoff.pdf", bbox_inches="tight")
        plt.show(); print("Saved → results/tradeoff.pdf")
    else:
        print("Need multi-budget results (e.g. h2o_10pct, h2o_20pct) for this plot.")

In [ ]:
# ── 4. Latency + macro-avg summary ───────────────────────────────────────────
if not df.empty:
    summary = (
        df.groupby("method")
          .agg(macro_score=("score", "mean"), avg_latency=("latency", "mean"))
          .round(2)
          .sort_values("macro_score", ascending=False)
    )
    display(summary)

    fig, ax = plt.subplots(figsize=(8, 4))
    lat  = summary["avg_latency"].sort_values()
    bars = ax.barh(lat.index, lat.values,
                   color=sns.color_palette("muted", n_colors=len(lat)))
    ax.bar_label(bars, fmt="%.2f s", padding=5)
    ax.set_xlabel("Average latency per sample (s)")
    ax.set_title("Inference latency by KV method")
    ax.set_xlim(0, lat.max() * 1.3)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "latency.pdf", bbox_inches="tight")
    plt.show(); print("Saved → results/latency.pdf")

---
## Step 8 — Discussion template

Fill in for your report using numbers from Step 7.

---

### 8.1 H2O

**Why it works:**  
Attention weights follow a power-law distribution — a handful of tokens accumulate the majority of attention mass. H2O exploits this by maintaining a running top-K of "heavy hitter" tokens per layer. The eviction is provably near-optimal under a submodular coverage objective (see §4 of the paper).

**Why it struggles on `[task]`:**  *(fill in)*  
Example: `passage_count` involves counting uniform paragraphs — no single token is semantically dominant, so cumulative scores are near-uniform and eviction is nearly random.

**Efficiency–accuracy trade-off:**  *(fill in after full eval)*

---

### 8.2 StreamingLLM

**Why it works:**  
Softmax always sums to 1. If no key is semantically relevant, the model has nowhere to "put" attention — it defaults to early tokens. Keeping those sink tokens prevents the attention distribution from collapsing. The sliding window preserves local context for fluent generation.

**Why it struggles on `[task]`:**  *(fill in)*  
Example: `2wikimqa` (multi-hop QA) — the relevant evidence often appears in the middle of the document, outside both the sink window and the recent window, so it gets evicted.

**Efficiency–accuracy trade-off:**  *(fill in after full eval)*

---

### 8.3 Comparison table  *(fill in)*

| | Full | H2O 20% | StreamingLLM 20% |
|---|---|---|---|
| Macro-avg score | — | — | — |
| Avg latency | — | — | — |
| Best task | — | — | — |
| Worst task | — | — | — |

---

### 8.4 Limitations & future work
- Both methods use a **fixed, uniform budget** per layer. Layer-adaptive budgets (e.g. PyramidKV assigns larger budgets to lower layers) can improve the trade-off.
- H2O's score accumulation adds CPU overhead; quantised score vectors could reduce this.
- SnapKV uses 2-D pooling on the prompt's attention map rather than cumulative decoding scores — better suited to tasks where the prompt fully determines which tokens matter.